# Project — Airline AI Assistant (Tool Calling)

**Reference guide:** give an LLM *tools* so it can look up (and later update) ticket prices instead of guessing.

Day 3 covered chat UIs, history, and system prompts. Those are still required here. What is new is the **tool-calling handshake**: the model does not run your Python. It *asks* you to run a function, you execute it, then you send the result back so it can answer the customer.

### What you'll learn

1. **Baseline FlightAI chatbot** — Gradio + system persona, no tools (prices will be invented)
2. **Tools as JSON Schema** — describe a Python function so the model can request it
3. **The tool loop** — `finish_reason == "tool_calls"` → run code → append `role: tool` → call the API again
4. **Parallel vs sequential tools** — several calls in one turn, then a `while` loop for follow-up calls
5. **SQLite backend** — swap the in-memory dict for a real store
6. **Write tool** — `set_ticket_price` so staff can update fares through chat

> Run cells top-to-bottom. Later cells redefine `chat`, `get_ticket_price`, and `tools`. Each `launch()` starts another local Gradio app (ports increment: 7860, 7861, …).

### Code review (what this notebook originally had wrong)

| Issue | Why it matters | Fix in this notebook |
|---|---|---|
| Tool schema description was truncated (`"The city that the customer wants to "`) | Vague parameter docs make the model pass the wrong argument | Complete the description |
| Schema missing `required` and `additionalProperties: False` | Model may omit `destination_city` or invent extra keys | Match the official function-tool schema |
| No `tools = [{"type": "function", "function": ...}]` wrapper | The API never sees the tool | Wrap every function dict |
| `ticket_prices.get(city)` had no default | Unknown cities returned `"... is None"` | Default to `"Unknown ticket price"` |
| Notebook stopped before the tool loop | The model could describe the tool but never *use* it | Full handshake, then parallel + sequential loops |
| Official `handle_tool_call` only handled `tool_calls[0]` and could `UnboundLocalError` on an unknown name | Multi-city questions fail; a mismatch crashes the UI | Loop all calls; dispatch table with a fallback |


## 1. Setup

Load the API key, create one OpenAI client, pick a model.

`json` is required later: the model returns tool arguments as a **JSON string**, not a Python dict.


In [ ]:
# --- Imports ---
# os          : read OPENAI_API_KEY from the environment
# json        : parse tool arguments the model returns as a JSON string
# sqlite3     : optional persistent store for ticket prices (section 10)
# load_dotenv : load a local .env file into os.environ
# OpenAI      : Chat Completions client (also used for tool calling)
# gradio      : ChatInterface UI — same contract as Day 3
import json
import os
import sqlite3
from collections.abc import Callable
from typing import Any, TypedDict

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


class ChatMessage(TypedDict):
    """One turn in the OpenAI / Gradio messages format."""
    role: str      # "system" | "user" | "assistant" | "tool"
    content: str


load_dotenv(override=True)

openai_api_key: str | None = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set — add OPENAI_API_KEY to your .env file")


### Client and model

`OpenAI()` reads `OPENAI_API_KEY`. Keep `MODEL` as a global so every `chat.completions.create(...)` stays in sync.

Swap the id if a model is unavailable. Tool calling needs a model that supports **function tools** (frontier chat models do; tiny local models sometimes do not).


In [ ]:
openai = OpenAI()

# Change this if your account does not expose this id.
# Day-3 default was gpt-4o-mini; the course notebook uses gpt-4.1-mini.
MODEL = "gpt-5.4-nano"

# Local alternative (Ollama must already be running):
# MODEL = "llama3.2"
# openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

print(f"Using model: {MODEL}")


## 2. System message (airline persona)

The system role is still the cheapest way to lock tone and honesty. Short answers make it obvious when the model is **guessing** a fare — that is the point of the next section.

We will extend this prompt later so the model knows *when* it is allowed to write prices.


In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
""".strip()


## 3. Baseline chatbot — no tools

Same pattern as Day 3:

```text
messages = [system] + history + [latest user turn]
```

**Try this in the UI:** "How much is a return ticket to London?"

The model has **no fare table**. A well-behaved prompt ("if you don't know, say so") may refuse; a weaker prompt will invent `$499` or similar. Either way, you cannot *trust* the number. Tools are how you replace guessing with a lookup.


In [ ]:
def project_history(history: list[dict[str, Any]]) -> list[ChatMessage]:
    """Keep only role + content. Gradio may attach extra keys we must not send."""
    return [{"role": h["role"], "content": h["content"]} for h in history]


def chat_no_tools(message: str, history: list[dict[str, Any]]) -> str | None:
    messages: list[dict[str, Any]] = (
        [{"role": "system", "content": system_message}]
        + project_history(history)
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


gr.ChatInterface(fn=chat_no_tools, type="messages").launch()


## 4. Tools — the model requests, your code executes

Frontier models can emit a structured **tool call** instead of (or before) a user-facing sentence.

Important mental model:

1. You send a **JSON Schema** that describes each function (`name`, `description`, `parameters`).
2. The model may reply with `finish_reason="tool_calls"` and one or more `{name, arguments, id}` records.
3. **Your Python** runs the matching function.
4. You append the assistant tool-call message **and** a `role: "tool"` result for each `tool_call_id`.
5. You call the API again. Now the model can answer with a real number.

The model never gets a shell. If you do not run the function, nothing happens on your machine. That is also why validation belongs in *your* function, not in the prompt.


## 5. The Python function (source of truth)

Keys are lowercase so `"London"` and `"london"` hit the same fare. Always `.lower()` the lookup.

**Bug that was here:** `.get(city)` with no default produced `"The price of a ticket to rome is None"`. The model would then confidently tell the customer the price is None. Return a clear unknown string instead.


In [ ]:
# In-memory catalog — later replaced by SQLite without changing the chat loop
ticket_prices = {
    "london": "$799",
    "paris": "$899",
    "tokyo": "$1400",
    "berlin": "$499",
}


def get_ticket_price(destination_city: str) -> str:
    """Look up a return-ticket price. Safe to call even if the city is unknown."""
    city = destination_city.lower()
    print(f"Tool called for city {city}")
    price = ticket_prices.get(city, "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


# Smoke tests — the UI is not required to know this function works
print(get_ticket_price("London"))
print(get_ticket_price("Rome"))  # should NOT say "is None"


## 6. Describe the function to the model (JSON Schema)

This dict is **not** Python. It is a JSON Schema the API forwards to the model.

| Field | Role |
|---|---|
| `name` | Must match the Python function you will dispatch on |
| `description` | The model uses this to decide *whether* to call the tool |
| `parameters.properties` | Argument names and types the model must fill in |
| `required` | Arguments that must be present |
| `additionalProperties: False` | Reject extra invented keys |

Then wrap it: `tools = [{"type": "function", "function": price_function}]`.

The course uses this older `tools` shape (still valid). Newer SDKs also have `client.responses` / `pydantic` helpers — same idea.


In [ ]:
price_function = {
    "name": "get_ticket_price",
    "description": (
        "Get the price of a return ticket to the destination city. "
        "Use this whenever the customer asks what a flight costs."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": price_function}]
tools


## 7. One tool round-trip

`chat.completions.create(..., tools=tools)` lets the model choose:

- ordinary text (`finish_reason="stop"`), or
- one or more tool calls (`finish_reason="tool_calls"`)

### Message list after a tool call

```text
system
...prior history...
user          "How much to London?"
assistant     content=None, tool_calls=[{id, name=get_ticket_price, arguments={"destination_city":"London"}}]
tool          tool_call_id=<same id>, content="The price of a ticket to London is $799"
assistant     "A return ticket to London is $799."
```

You must append the **assistant tool-call message object** the API returned (not a homemade dict). Then append one `role: "tool"` message per call. The second `create` is a normal completion so the model can phrase the answer.

**Name shadowing:** do not reuse the parameter `message` for `response.choices[0].message`. That makes the user text harder to debug. Use `assistant_msg`.


In [ ]:
def handle_tool_call(assistant_msg: Any) -> dict[str, str]:
    """Run the first tool call only. Kept as a stepping stone — see handle_tool_calls next."""
    tool_call = assistant_msg.tool_calls[0]
    if tool_call.function.name != "get_ticket_price":
        return {
            "role": "tool",
            "content": f"Unknown tool: {tool_call.function.name}",
            "tool_call_id": tool_call.id,
        }
    arguments = json.loads(tool_call.function.arguments)
    city = arguments.get("destination_city", "")
    return {
        "role": "tool",
        "content": get_ticket_price(city),
        "tool_call_id": tool_call.id,
    }


def chat_one_tool_round(message: str, history: list[dict[str, Any]]) -> str | None:
    messages: list[Any] = (
        [{"role": "system", "content": system_message}]
        + project_history(history)
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )

    if response.choices[0].finish_reason == "tool_calls":
        assistant_msg = response.choices[0].message
        tool_result = handle_tool_call(assistant_msg)
        messages.append(assistant_msg)
        messages.append(tool_result)
        # Second call: the model now has the price and can speak to the customer.
        # tools= is omitted so this round is "answer the user", not "call again".
        response = openai.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content


## 8. Several tool calls in one assistant message

Ask: "What do tickets to London and Paris cost?"

The model often returns **two** `tool_calls` in the same message. Handling only `[0]` answers London and silently drops Paris.

`handle_tool_calls` loops every call and returns a list. `messages.extend(responses)` keeps `tool_call_id`s aligned with the assistant message.

Still **one** extra API round after that — both prices go back together.


In [ ]:
def handle_tool_calls(assistant_msg: Any) -> list[dict[str, str]]:
    """Execute every tool call in this assistant message (parallel batch)."""
    responses: list[dict[str, str]] = []
    for tool_call in assistant_msg.tool_calls or []:
        name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        if name == "get_ticket_price":
            city = arguments.get("destination_city", "")
            content = get_ticket_price(city)
        else:
            content = f"Unknown tool: {name}"
        responses.append(
            {
                "role": "tool",
                "content": content,
                "tool_call_id": tool_call.id,
            }
        )
    return responses


def chat_parallel_tools(message: str, history: list[dict[str, Any]]) -> str | None:
    messages: list[Any] = (
        [{"role": "system", "content": system_message}]
        + project_history(history)
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )

    if response.choices[0].finish_reason == "tool_calls":
        assistant_msg = response.choices[0].message
        tool_results = handle_tool_calls(assistant_msg)
        messages.append(assistant_msg)
        messages.extend(tool_results)
        response = openai.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content


## 9. Sequential tools — keep offering tools in a `while` loop

One extra round is not always enough. After seeing London's price the model may decide it also needs Paris, or (later) it may `get` then `set`.

Change `if` to `while`, and **pass `tools=tools` on every follow-up call** so the model can request another function. Cap the loop so a confused model cannot spin forever.

This is the production-shaped chat function. The Gradio app below uses it.


In [ ]:
MAX_TOOL_ROUNDS = 5


def chat(message: str, history: list[dict[str, Any]]) -> str | None:
    messages: list[Any] = (
        [{"role": "system", "content": system_message}]
        + project_history(history)
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )

    rounds = 0
    while response.choices[0].finish_reason == "tool_calls":
        rounds += 1
        if rounds > MAX_TOOL_ROUNDS:
            break
        assistant_msg = response.choices[0].message
        tool_results = handle_tool_calls(assistant_msg)
        messages.append(assistant_msg)
        messages.extend(tool_results)
        response = openai.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )

    return response.choices[0].message.content


gr.ChatInterface(fn=chat, type="messages").launch()


**Prompts to try**

- "How much is a return to London?" → one tool call, then a short sentence with `$799`
- "Compare London and Tokyo" → two tool calls in one turn
- "How much to Mars?" → tool runs, returns unknown, assistant should not invent a fare
- "What's your baggage policy?" → no tool; the model should admit it does not know

Watch the notebook stdout: `Tool called for city ...` proves the handshake ran. If that line never prints, the model answered from weights alone.


## 10. Persist prices in SQLite

An in-memory dict dies when the kernel restarts. A tiny SQLite file is enough to show that **tools are an interface over any backend** (dict, DB, HTTP API, booking system).

Notes:

- `CREATE TABLE IF NOT EXISTS` so re-running the cell is safe
- Parameterized `?` placeholders — never interpolate the city into SQL
- `city TEXT PRIMARY KEY` plus `.lower()` keeps one row per destination
- `get_ticket_price` is **redefined** here; the chat loop does not change because it still calls the same name


In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    conn.execute(
        "CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)"
    )
    conn.commit()


def get_ticket_price(destination_city: str) -> str:
    print(f"DATABASE TOOL CALLED: Getting price for {destination_city}", flush=True)
    with sqlite3.connect(DB) as conn:
        row = conn.execute(
            "SELECT price FROM prices WHERE city = ?",
            (destination_city.lower(),),
        ).fetchone()
    if row is None:
        return f"No price data available for {destination_city}"
    return f"The price of a ticket to {destination_city} is ${row[0]}"


def set_ticket_price(destination_city: str, price: float) -> str:
    print(
        f"DATABASE TOOL CALLED: Setting price for {destination_city} to {price}",
        flush=True,
    )
    with sqlite3.connect(DB) as conn:
        conn.execute(
            """
            INSERT INTO prices (city, price) VALUES (?, ?)
            ON CONFLICT(city) DO UPDATE SET price = excluded.price
            """,
            (destination_city.lower(), price),
        )
        conn.commit()
    return f"Ticket price to {destination_city} is now ${price}"


Seed the database, then confirm a lookup. Until you seed, `get_ticket_price("London")` correctly returns *no price data* — that is the empty-state you want, not a hallucinated fare.


In [ ]:
print("Before seed:", get_ticket_price("London"))

seed_prices = {"london": 799, "paris": 899, "tokyo": 1420, "berlin": 499, "sydney": 2999}
for city, price in seed_prices.items():
    set_ticket_price(city, price)

print("After seed:", get_ticket_price("London"))
print(get_ticket_price("Tokyo"))


## 11. Exercise — add a `set_ticket_price` tool

The course exercise: expose the write function as a second tool so an agent can update fares in chat.

What to add:

1. A second schema with **two** required properties (`destination_city`, `price`)
2. Both schemas in `tools`
3. A dispatcher so `handle_tool_calls` does not grow a long `if/elif`
4. A tighter system prompt: only set a price when the user clearly supplies city **and** amount

The `while` loop from Part 9 already supports get-then-set or set-then-get in one user turn.


In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.

You can look up ticket prices with get_ticket_price.
You can update ticket prices with set_ticket_price only when the user clearly
provides both a destination city and a numeric price. Never invent a new fare.
""".strip()

set_price_function = {
    "name": "set_ticket_price",
    "description": (
        "Set or update the return-ticket price for a destination city. "
        "Call this only when the user explicitly gives both a city and a price."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The destination city whose ticket price should be updated",
            },
            "price": {
                "type": "number",
                "description": "The new ticket price in US dollars",
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function},
]

# name → callable that accepts the parsed arguments dict
TOOL_HANDLERS: dict[str, Callable[[dict[str, Any]], str]] = {
    "get_ticket_price": lambda args: get_ticket_price(args["destination_city"]),
    "set_ticket_price": lambda args: set_ticket_price(
        args["destination_city"], args["price"]
    ),
}


def handle_tool_calls(assistant_msg: Any) -> list[dict[str, str]]:
    responses: list[dict[str, str]] = []
    for tool_call in assistant_msg.tool_calls or []:
        name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        handler = TOOL_HANDLERS.get(name)
        content = handler(arguments) if handler else f"Unknown tool: {name}"
        responses.append(
            {
                "role": "tool",
                "content": content,
                "tool_call_id": tool_call.id,
            }
        )
    return responses


`chat` is unchanged — it already loops and passes `tools`. Re-launch so Gradio picks up the new `system_message`, `tools`, and `handle_tool_calls`.


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()


**Prompts to try (write path)**

- "Set the Tokyo fare to 1500" → stdout shows `Setting price...`, then a confirmation
- "How much to Tokyo now?" → should reflect `$1500.0` (SQLite stores REAL)
- "Make Paris cheaper" with no number → should **not** call `set_ticket_price`

`prices.db` is created in this notebook's folder. Delete it to reset the catalog.


## Quick reference (cheat sheet)

| Piece | Role |
|---|---|
| `system_message` | Persona + when to use each tool |
| `get_ticket_price` / `set_ticket_price` | Real Python; this is the only code that touches data |
| `price_function` / `set_price_function` | JSON Schema the model sees |
| `tools=[{"type":"function","function": ...}]` | API payload wrapping those schemas |
| `finish_reason == "tool_calls"` | Model wants you to run something |
| `assistant_msg.tool_calls` | List of `{id, function.name, function.arguments}` |
| `json.loads(arguments)` | Arguments arrive as a **string** |
| `role: "tool"` + `tool_call_id` | Your result, tied to that call |
| `if` one extra `create` | One batch of parallel calls, then answer |
| `while` + `tools=tools` every time | Follow-up calls (get then set, etc.) |
| `MAX_TOOL_ROUNDS` | Safety cap |
| `TOOL_HANDLERS` | Add a tool without rewriting the loop |
| SQLite `?` placeholders | Backend behind the same function names |

### Typical request shape

```python
response = openai.chat.completions.create(
    model=MODEL,
    messages=messages,   # system + history + user [+ assistant tool calls + tool results]
    tools=tools,         # omit only when you want to force a final text answer
)
```

### Logic pitfalls to remember

1. **Hallucinated fares** — without tools (or if you forget `tools=`), the model invents numbers.
2. **`None` prices** — always give `.get` / SQL a real unknown path.
3. **Incomplete schema** — truncated descriptions and missing `required` cause bad arguments.
4. **Only handling `tool_calls[0]`** — multi-city questions lose cities.
5. **Shadowing `message`** — keep the user string and the assistant message as different names.
6. **Forgetting `tools=` in the `while` body** — the model cannot request a second function.
7. **SQL injection** — tools pass model-chosen strings; use bound parameters.
8. **Write tools are dangerous** — a `set_*` tool is a mutation API. Constrain it in the schema *and* the system prompt.

### Business takeaway

This is the jump from a chatbot that *talks about* your business to an assistant that **does** something: read a catalog, write a row, eventually call a booking API. The LLM is the language layer. Your functions are the system of record. Keep that boundary sharp: validate in Python, describe in schema, never let the model touch the database directly.


## Conclusion

Day 4 is the tool-calling loop, not the airline theme.

You started with a polite FlightAI that could not know fares. You then published a Python lookup as a JSON Schema tool, ran whatever the model requested, and fed the result back so the customer heard a real price. Parallel calls cover "London and Paris" in one turn; a `while` loop covers follow-up calls. Swapping the dict for SQLite (and adding `set_ticket_price`) shows the same handshake sitting on a real store.

Reuse this notebook as a template: copy `chat` + `handle_tool_calls` + `TOOL_HANDLERS`, replace the functions and schemas, keep the message protocol identical. For production, add auth on write tools, log every call, and treat model arguments as untrusted input.
